In [1]:
import sys
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.home() / "si630_nlp_project"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    PROCESSED_DATA_DIR,
    BASELINE_MODELS_DIR,
    REPORTS_DIR,
    TABLE_DIR,
)

print(PROJECT_ROOT)
print(PROCESSED_DATA_DIR)

/home/xinyihu/si630_nlp_project
/home/xinyihu/si630_nlp_project/data/processed


In [3]:
DATASET_TAG = "large_lite"
EXPERIMENT_TAG = "finbert_finetune_v1"

TEXT_COL = "full_text"
LABEL_COL = "label_5d"

MODEL_NAME = "ProsusAI/finbert"

MAX_LENGTH = 512
CHUNK_BODY_LEN = 510
MAX_CHUNKS_PER_DOC = 8

SEED = 42
DEBUG_SMALL = False

TRAIN_SUBSAMPLE_DOCS = None

# training
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [4]:
BASELINE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

FINETUNE_MODEL_DIR = BASELINE_MODELS_DIR / f"finbert_finetuned_{DATASET_TAG}_{EXPERIMENT_TAG}"
REPORT_PATH = REPORTS_DIR / f"finbert_finetuned_{DATASET_TAG}_{EXPERIMENT_TAG}_results.json"
VAL_TABLE_PATH = TABLE_DIR / f"finbert_finetuned_{DATASET_TAG}_{EXPERIMENT_TAG}_val_doc_metrics.csv"
TEST_TABLE_PATH = TABLE_DIR / f"finbert_finetuned_{DATASET_TAG}_{EXPERIMENT_TAG}_test_doc_metrics.csv"

CHUNK_STATS_PATH = REPORTS_DIR / f"finbert_finetuned_{DATASET_TAG}_{EXPERIMENT_TAG}_chunk_stats.json"

print(FINETUNE_MODEL_DIR)
print(REPORT_PATH)

/home/xinyihu/si630_nlp_project/models/baselines/finbert_finetuned_large_lite_finbert_finetune_v1
/home/xinyihu/si630_nlp_project/outputs/reports/finbert_finetuned_large_lite_finbert_finetune_v1_results.json


In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [6]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_doc_5d_large_lite.parquet")
val_df = pd.read_parquet(PROCESSED_DATA_DIR / "validation_doc_5d_large_lite.parquet")
test_df = pd.read_parquet(PROCESSED_DATA_DIR / "test_doc_5d_large_lite.parquet")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COL].value_counts(dropna=False).sort_index())

print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts(dropna=False).sort_index())

print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts(dropna=False).sort_index())

Train: (52411, 8)
Validation: (866, 8)
Test: (1819, 8)

Train label distribution:
label_5d
0    24132
1    28279
Name: count, dtype: int64

Validation label distribution:
label_5d
0    368
1    498
Name: count, dtype: int64

Test label distribution:
label_5d
0     784
1    1035
Name: count, dtype: int64


In [7]:
if DEBUG_SMALL:
    train_df = train_df.head(300)
    val_df = val_df.head(80)
    test_df = test_df.head(120)
    print("DEBUG_SMALL is ON")

if TRAIN_SUBSAMPLE_DOCS is not None:
    train_df = train_df.sample(TRAIN_SUBSAMPLE_DOCS, random_state=SEED).reset_index(drop=True)
    print(f"Using TRAIN_SUBSAMPLE_DOCS = {TRAIN_SUBSAMPLE_DOCS}")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(train_df.shape, val_df.shape, test_df.shape)

(52411, 8) (866, 8) (1819, 8)


In [8]:
train_df["doc_internal_id"] = [f"train_{i}" for i in range(len(train_df))]
val_df["doc_internal_id"] = [f"val_{i}" for i in range(len(val_df))]
test_df["doc_internal_id"] = [f"test_{i}" for i in range(len(test_df))]

train_df[[TEXT_COL, LABEL_COL, "doc_internal_id"]].head()

,full_text,label_5d,doc_internal_id
0,ITEM 1.BUSINESS General AAR CORP. and its subs...,1,train_0
1,ITEM 1. BUSINESS General AAR CORP. and its sub...,1,train_1
2,ITEM 1. BUSINESS General AAR CORP. and its sub...,1,train_2
3,ITEM 1. BUSINESS General AAR CORP. and its sub...,0,train_3
4,ITEM 1. BUSINESS (Dollars in millions) General...,1,train_4


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.unk_token

print(type(tokenizer))

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


In [10]:
def chunk_document(text: str, tokenizer, max_chunks_per_doc=8, chunk_body_len=480):
    text = str(text)
    tokens = tokenizer.tokenize(text)

    chunks = []
    for i in range(0, len(tokens), chunk_body_len):
        if len(chunks) >= max_chunks_per_doc:
            break
        chunk_tokens = tokens[i:i + chunk_body_len]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)
        chunks.append(chunk_text)

    if len(chunks) == 0:
        chunks = [""]

    return chunks

In [11]:
from tqdm import tqdm

def build_chunk_dataframe(df, split_name):
    records = []
    chunk_counts = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Building chunks for {split_name}"):
        doc_id = row["doc_internal_id"]
        label = int(row[LABEL_COL])
        text = row[TEXT_COL]

        chunks = chunk_document(
            text=text,
            tokenizer=tokenizer,
            max_chunks_per_doc=MAX_CHUNKS_PER_DOC,
            chunk_body_len=CHUNK_BODY_LEN,
        )

        chunk_counts.append(len(chunks))

        for chunk_idx, chunk_text in enumerate(chunks):
            records.append({
                "doc_internal_id": doc_id,
                "chunk_id": f"{doc_id}_chunk_{chunk_idx}",
                "text": chunk_text,
                "label": label,
                "split": split_name,
            })

    chunk_df = pd.DataFrame(records)
    return chunk_df, chunk_counts

In [12]:
train_chunk_df, train_chunk_counts = build_chunk_dataframe(train_df, "train")
print("Train chunk df:", train_chunk_df.shape)
print("Avg train chunks per doc:", np.mean(train_chunk_counts))

Building chunks for train: 100%|██████████| 52411/52411 [1:04:52<00:00, 13.47it/s]


Train chunk df: (418125, 5)
Avg train chunks per doc: 7.977810001717197


In [13]:
val_chunk_df, val_chunk_counts = build_chunk_dataframe(val_df, "validation")
print("Validation chunk df:", val_chunk_df.shape)
print("Avg val chunks per doc:", np.mean(val_chunk_counts))

Building chunks for validation: 100%|██████████| 866/866 [01:59<00:00,  7.24it/s]

Validation chunk df: (6928, 5)
Avg val chunks per doc: 8.0


In [14]:
test_chunk_df, test_chunk_counts = build_chunk_dataframe(test_df, "test")
print(test_chunk_df.shape)
print(np.mean(test_chunk_counts))

Building chunks for test: 100%|██████████| 1819/1819 [03:38<00:00,  8.32it/s]

(14532, 5)
7.989004947773502


In [15]:
chunk_stats = {
    "dataset_tag": DATASET_TAG,
    "experiment_tag": EXPERIMENT_TAG,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "max_chunks_per_doc": MAX_CHUNKS_PER_DOC,
    "avg_train_chunks_per_doc": float(np.mean(train_chunk_counts)),
    "avg_validation_chunks_per_doc": float(np.mean(val_chunk_counts)),
    "avg_test_chunks_per_doc": float(np.mean(test_chunk_counts)),
    "n_train_docs": int(len(train_df)),
    "n_validation_docs": int(len(val_df)),
    "n_test_docs": int(len(test_df)),
    "n_train_chunks": int(len(train_chunk_df)),
    "n_validation_chunks": int(len(val_chunk_df)),
    "n_test_chunks": int(len(test_chunk_df)),
}

with open(CHUNK_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(chunk_stats, f, indent=2)

chunk_stats

{'dataset_tag': 'large_lite',
 'experiment_tag': 'finbert_finetune_v1',
 'model_name': 'ProsusAI/finbert',
 'max_length': 512,
 'max_chunks_per_doc': 8,
 'avg_train_chunks_per_doc': 7.977810001717197,
 'avg_validation_chunks_per_doc': 8.0,
 'avg_test_chunks_per_doc': 7.989004947773502,
 'n_train_docs': 52411,
 'n_validation_docs': 866,
 'n_test_docs': 1819,
 'n_train_chunks': 418125,
 'n_validation_chunks': 6928,
 'n_test_chunks': 14532}

In [16]:
train_ds = Dataset.from_pandas(train_chunk_df[["doc_internal_id", "chunk_id", "text", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_chunk_df[["doc_internal_id", "chunk_id", "text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_chunk_df[["doc_internal_id", "chunk_id", "text", "label"]], preserve_index=False)

train_ds, val_ds, test_ds

(Dataset({
     features: ['doc_internal_id', 'chunk_id', 'text', 'label'],
     num_rows: 418125
 }),
 Dataset({
     features: ['doc_internal_id', 'chunk_id', 'text', 'label'],
     num_rows: 6928
 }),
 Dataset({
     features: ['doc_internal_id', 'chunk_id', 'text', 'label'],
     num_rows: 14532
 }))

In [17]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])

train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")
test_tok = test_tok.rename_column("label", "labels")

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

Map: 100%|██████████| 14532/14532 [00:03<00:00, 4382.87 examples/s]


In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

model.to(DEVICE)
print(type(model))
print(model.config.num_labels)

You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 49734.83it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |                                                                                       
-----------------------------+------------+---------------------------------------------------------------------------------------
bert.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.bias              | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.weight            | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MI

<class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'>
2


In [19]:
def compute_chunk_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
    }

In [20]:
training_args = TrainingArguments(
    output_dir=str(FINETUNE_MODEL_DIR),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

In [21]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_chunk_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [22]:
train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.764666,0.682910,0.575058,0.730205
2,2.732954,0.696789,0.548499,0.657618


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=52266, training_loss=2.752751512395726, metrics={'train_runtime': 3287.8666, 'train_samples_per_second': 254.344, 'train_steps_per_second': 15.897, 'total_flos': 2.200266200448e+17, 'train_loss': 2.752751512395726, 'epoch': 2.0})

In [23]:
trainer.save_model(str(FINETUNE_MODEL_DIR))
tokenizer.save_pretrained(str(FINETUNE_MODEL_DIR))
print("Saved fine-tuned model to:", FINETUNE_MODEL_DIR)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.80s/it]

Saved fine-tuned model to: /home/xinyihu/si630_nlp_project/models/baselines/finbert_finetuned_large_lite_finbert_finetune_v1


In [24]:
def document_level_predictions(trainer, tokenized_ds, original_chunk_df, threshold=0.5):
    pred_output = trainer.predict(tokenized_ds)
    logits = pred_output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    pos_probs = probs[:, 1]

    temp_df = original_chunk_df.copy().reset_index(drop=True)
    temp_df["pos_prob"] = pos_probs

    doc_prob_df = (
        temp_df.groupby("doc_internal_id", as_index=False)["pos_prob"]
        .mean()
        .rename(columns={"pos_prob": "doc_pos_prob"})
    )

    doc_prob_df["pred_label"] = (doc_prob_df["doc_pos_prob"] >= threshold).astype(int)

    return doc_prob_df

In [25]:
val_doc_probs = document_level_predictions(
    trainer=trainer,
    tokenized_ds=val_tok,
    original_chunk_df=val_chunk_df,
    threshold=0.5,
)

val_doc_gold = val_df[["doc_internal_id", LABEL_COL]].rename(columns={LABEL_COL: "true_label"})
val_doc_eval = val_doc_gold.merge(val_doc_probs, on="doc_internal_id", how="left")

val_acc = accuracy_score(val_doc_eval["true_label"], val_doc_eval["pred_label"])
val_f1 = f1_score(val_doc_eval["true_label"], val_doc_eval["pred_label"])
val_cm = confusion_matrix(val_doc_eval["true_label"], val_doc_eval["pred_label"])
val_report = classification_report(val_doc_eval["true_label"], val_doc_eval["pred_label"], output_dict=True)

print("=== VALIDATION DOCUMENT-LEVEL PERFORMANCE ===")
print("Accuracy:", round(val_acc, 4))
print("F1:", round(val_f1, 4))
print("Confusion Matrix:")
print(val_cm)

=== VALIDATION DOCUMENT-LEVEL PERFORMANCE ===
Accuracy: 0.5751
F1: 0.7302
Confusion Matrix:
[[  0 368]
 [  0 498]]


/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

In [26]:
test_doc_probs = document_level_predictions(
    trainer=trainer,
    tokenized_ds=test_tok,
    original_chunk_df=test_chunk_df,
    threshold=0.5,
)

test_doc_gold = test_df[["doc_internal_id", LABEL_COL]].rename(columns={LABEL_COL: "true_label"})
test_doc_eval = test_doc_gold.merge(test_doc_probs, on="doc_internal_id", how="left")

test_acc = accuracy_score(test_doc_eval["true_label"], test_doc_eval["pred_label"])
test_f1 = f1_score(test_doc_eval["true_label"], test_doc_eval["pred_label"])
test_cm = confusion_matrix(test_doc_eval["true_label"], test_doc_eval["pred_label"])
test_report = classification_report(test_doc_eval["true_label"], test_doc_eval["pred_label"], output_dict=True)

print("=== TEST DOCUMENT-LEVEL PERFORMANCE ===")
print("Accuracy:", round(test_acc, 4))
print("F1:", round(test_f1, 4))
print("Confusion Matrix:")
print(test_cm)

=== TEST DOCUMENT-LEVEL PERFORMANCE ===
Accuracy: 0.569
F1: 0.7253
Confusion Matrix:
[[   0  784]
 [   0 1035]]


/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/xinyihu/.conda/envs/si630finbert/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

In [27]:
val_doc_eval.to_csv(VAL_TABLE_PATH, index=False)
test_doc_eval.to_csv(TEST_TABLE_PATH, index=False)

print(VAL_TABLE_PATH)
print(TEST_TABLE_PATH)

/home/xinyihu/si630_nlp_project/outputs/tables/finbert_finetuned_large_lite_finbert_finetune_v1_val_doc_metrics.csv
/home/xinyihu/si630_nlp_project/outputs/tables/finbert_finetuned_large_lite_finbert_finetune_v1_test_doc_metrics.csv


In [28]:
payload = {
    "dataset_tag": DATASET_TAG,
    "experiment_tag": EXPERIMENT_TAG,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "num_train_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "max_length": MAX_LENGTH,
    "max_chunks_per_doc": MAX_CHUNKS_PER_DOC,
    "n_train_docs": int(len(train_df)),
    "n_validation_docs": int(len(val_df)),
    "n_test_docs": int(len(test_df)),
    "n_train_chunks": int(len(train_chunk_df)),
    "n_validation_chunks": int(len(val_chunk_df)),
    "n_test_chunks": int(len(test_chunk_df)),
    "validation_doc_level": {
        "accuracy": float(val_acc),
        "f1": float(val_f1),
        "confusion_matrix": val_cm.tolist(),
        "classification_report": val_report,
    },
    "test_doc_level": {
        "accuracy": float(test_acc),
        "f1": float(test_f1),
        "confusion_matrix": test_cm.tolist(),
        "classification_report": test_report,
    },
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("Saved report to:", REPORT_PATH)
payload

Saved report to: /home/xinyihu/si630_nlp_project/outputs/reports/finbert_finetuned_large_lite_finbert_finetune_v1_results.json


{'dataset_tag': 'large_lite',
 'experiment_tag': 'finbert_finetune_v1',
 'model_name': 'ProsusAI/finbert',
 'device': 'cuda',
 'num_train_epochs': 2,
 'learning_rate': 2e-05,
 'weight_decay': 0.01,
 'train_batch_size': 4,
 'eval_batch_size': 8,
 'gradient_accumulation_steps': 4,
 'max_length': 512,
 'max_chunks_per_doc': 8,
 'n_train_docs': 52411,
 'n_validation_docs': 866,
 'n_test_docs': 1819,
 'n_train_chunks': 418125,
 'n_validation_chunks': 6928,
 'n_test_chunks': 14532,
 'validation_doc_level': {'accuracy': 0.5750577367205543,
  'f1': 0.7302052785923754,
  'confusion_matrix': [[0, 368], [0, 498]],
  'classification_report': {'0': {'precision': 0.0,
    'recall': 0.0,
    'f1-score': 0.0,
    'support': 368.0},
   '1': {'precision': 0.5750577367205543,
    'recall': 1.0,
    'f1-score': 0.7302052785923754,
    'support': 498.0},
   'accuracy': 0.5750577367205543,
   'macro avg': {'precision': 0.28752886836027713,
    'recall': 0.5,
    'f1-score': 0.3651026392961877,
    'support'